In [ ]:
# [Cell 1: Environment Setup & Data Loading]
"""
@brief Setup environment and load MNIST dataset.
@details Loads preprocessed MNIST data and normalizes pixel values 
         to a range of [0, 1] for hardware-compatible inference.
"""
import numpy as np
import time
from scipy.signal import convolve2d
from tqdm import trange

# Load MNIST dataset (Expected: 'mnist-original.npy' in the same directory)
mnist = np.load("mnist-original.npy", allow_pickle= True)

# Data Preprocessing: Transpose and normalize to float32
X = mnist.item().get("data").astype(np.float32).T / 255
y = mnist.item().get("label")[0]

In [ ]:
from pynq import Overlay, allocate

# 1. Load the FPGA bitstream and initialize DMA interfaces
overlay = Overlay("bnn_accel.bit") 
dma_input = overlay.axi_dma_0
dma_output = overlay.axi_dma_1

In [3]:
overlay?

In [ ]:
# 2. Define hardware parameters
IN_H, IN_W = 28, 28
FC_OUT = 10

In [ ]:
batch_size = 10000

in_buffer = allocate(shape=(batch_size, IN_H, IN_W), dtype=np.float32)
out_buffer = allocate(shape=(batch_size, FC_OUT), dtype=np.uint32)

def feed_forward(X0, batch_size):
    np.copyto(in_buffer, X0.reshape(batch_size, IN_H , IN_W)) 
    
    dma_input.sendchannel.transfer(in_buffer)     
    dma_output.recvchannel.transfer(out_buffer)     

    dma_input.sendchannel.wait()
    dma_output.recvchannel.wait()

    return out_buffer

In [ ]:
st = time.perf_counter()

correct = 0; total = 0
pred_arr = np.zeros((70000, 10), dtype=np.uint32)  

for idx in range(len(X)//batch_size):
    xs = X[batch_size * idx : batch_size * (idx + 1)]

    outputs = feed_forward(xs, batch_size)
    pred_arr[batch_size*idx:batch_size*(idx+1)] = outputs
    
prediction = np.argmax(pred_arr, axis=1)
correct = np.sum(prediction == y)

et = time.perf_counter()

accuracy = (correct / len(X)) * 100
print(f"Final Accuracy: {accuracy:.2f}%, Time: {et - st:.3f} sec")

Final Accuracy: 94.80%, Time: 1.059 sec


In [ ]:
import time
import numpy as np
from tqdm import trange

st = time.perf_counter()

correct = 0; total = 0

progress_bar = trange(len(X) // batch_size)
pred_arr = np.zeros((70000, 10), dtype=np.uint32) 

for idx in progress_bar:
    xs = X[batch_size * idx : batch_size * (idx + 1)]

    outputs = feed_forward(xs, batch_size)
    pred_arr[batch_size*idx:batch_size*(idx+1)] = outputs

prediction = np.argmax(pred_arr, axis=1)

correct = np.sum(prediction == y)
accuracy = (correct / len(X)) * 100

et = time.perf_counter()

print(f"Final Accuracy: {accuracy:.2f}%, Time: {et - st:.3f} sec")

100%|██████████| 7/7 [00:01<00:00,  6.33it/s]

Final Accuracy: 94.80%, Time: 1.151 sec


In [ ]:
import time

st = time.perf_counter()

dma_input.sendchannel.transfer(in_buffer)    
dma_output.recvchannel.transfer(out_buffer)     

dma_input.sendchannel.wait()
dma_output.recvchannel.wait()

et = time.perf_counter()

print(f"Time: {((et-st)):.3f} s!")

Time: 0.082 s!


In [ ]:
import numpy as np
from line_profiler import LineProfiler

profiler = LineProfiler()
profiler.add_function(feed_forward)

def feed_forward_test(X0, ys, batch_size):
    np.copyto(in_buffer, X0.reshape(batch_size, IN_H , IN_W)   )
    #in_buffer[:] = X0.reshape(batch_size, IN_H, IN_W)    
    
    dma_input.sendchannel.transfer(in_buffer)     
    dma_output.recvchannel.transfer(out_buffer)     

    dma_input.sendchannel.wait()
    dma_output.recvchannel.wait()

    prediction = np.argmax(out_buffer, axis=1)
    correct = np.sum(prediction == ys)
    
    return correct

example_input = np.random.rand(10000, 28, 28).astype(np.float32)

for _ in range(10):
    profiler(feed_forward_test)(xs, ys, batch_size)

profiler.print_stats()

Timer unit: 1e-09 s

Total time: 0.685343 s
File: <ipython-input-10-e8e466451152>
Function: feed_forward_test at line 8

Line #      Hits         Time  Per Hit   % Time  Line Contents
     8                                           def feed_forward_test(X0, ys, batch_size):
     9         1   63516439.0    6e+07      9.3      np.copyto(in_buffer, X0.reshape(batch_size, IN_H , IN_W)   )
    10                                               #in_buffer[:] = X0.reshape(batch_size, IN_H, IN_W)    
    11                                               
    12         1     648311.0 648311.0      0.1      dma_input.sendchannel.transfer(in_buffer)     # 100*28*28*4
    13         1     294960.0 294960.0      0.0      dma_output.recvchannel.transfer(out_buffer)     # 100*10*4
    14                                           
    15         1  607793018.0    6e+08     88.7      dma_input.sendchannel.wait()
    16         1     559082.0 559082.0      0.1      dma_output.recvchannel.wait()
    17  

In [ ]:
import time
import numpy as np
from tqdm import trange

batch_size = 200
st = time.perf_counter()

correct = 0; total = 0

progress_bar = trange(len(X) // batch_size)

for idx in progress_bar:
    xs = X[batch_size * idx : batch_size * (idx + 1)]
    ys = y[batch_size * idx : batch_size * (idx + 1)]

    outputs = double_buffer_forward(xs, batch_size)

    prediction = np.argmax(outputs, axis=1)
    correct += np.sum(prediction == ys)
    total += batch_size

    accuracy = (correct / total) * 100
    progress_bar.set_postfix(accuracy=f"{accuracy:.2f}%")

et = time.perf_counter()

print(f"Final Accuracy: {accuracy:.2f}%, Time: {et - st:.3f} sec")

100%|██████████| 350/350 [01:08<00:00,  5.14it/s, accuracy=94.80%]

Final Accuracy: 94.80%, Time: 68.128 sec
